In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cmocean.cm as cmo
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.feature import NaturalEarthFeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER
import dask
import intake
#dask.config.set({"logging": "error"})
from pathlib import Path
import glob
from rechunker import rechunk
import os
import shutil

import sys
sys.path.append('access-om2-sst-budget/')
import bipolarMhwToolBox as MHW

#import cosima_cookbook as cc
import pandas as pd

%matplotlib inline

In [2]:
from dask.distributed import Client

client = Client()
client

/g/data/xp65/public/apps/med_conda/envs/analysis3-25.07/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 40039 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/40039/status,
Dashboard: /proxy/40039/status,Workers: 7
Total threads: 14,Total memory: 510.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:39537,Workers: 0
Dashboard: /proxy/40039/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:40311,Total threads: 2
Dashboard: /proxy/41655/status,Memory: 72.86 GiB
Nanny: tcp://127.0.0.1:39647,


In [3]:
# === Setup ===
base = '/scratch/e14/rmh561/access-om2/archive/025deg_jra55_iaf_cycle6_online_mlt/'
file_pattern = base + "output*/ocean/ocean_daily.nc"
file_list = sorted(glob.glob(file_pattern))

if not file_list:
    raise FileNotFoundError("No matching ocean_daily.nc files found!")

output_dir = Path("/scratch/m35/nm5072/TAS_thresholds/")
output_dir.mkdir(parents=True, exist_ok=True)

zarr_path = output_dir / "SH_MLD_ready.zarr"

In [4]:
# === Step 1: Preprocess and save to Zarr ===
print("🔄 Preprocessing and saving intermediate file to Zarr...")

ds = xr.open_mfdataset(
    file_list,
    combine="nested",
    concat_dim="time",
    compat="override",
    data_vars=['temp_in_mld'],
    coords="all",
    parallel=True,
    chunks={},  # no chunking at load
)

# Subset data to region and time
temp_mld = ds["temp_in_mld"].sel(
    yt_ocean=slice(-90, 0),
    time=slice("1989-01-01", "2018-12-31")
)

# Convert to degrees Celsius and rename
temp = (temp_mld / 1035).rename("temp")

🔄 Preprocessing and saving intermediate file to Zarr...


/g/data/xp65/public/apps/med_conda/envs/analysis3-25.07/lib/python3.11/site-packages/dask/_task_spec.py:763: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  return self.func(*new_argspec, **kwargs)
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.07/lib/python3.11/site-packages/dask/_task_spec.py:763: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instea

In [ ]:
# # Wrap in dataset and chunk
# ds = temp.to_dataset()
# ds = ds.chunk({"time": -1, "yt_ocean": 110, "xt_ocean": 82})

# # Save to Zarr
# ds.to_zarr(zarr_path, mode="w")
# print(f"✅ Saved intermediate Zarr file to: {zarr_path}")

# === Clean up any existing stores ===
for path in [zarr_path, f"{zarr_path}.tmp"]:
    if os.path.exists(path):
        print(f"🗑️  Removing existing {path}")
        shutil.rmtree(path)

# === OR: Use rechunker for efficient rechunking ===
print("💾 Rechunking with rechunker...")
ds_input = temp.to_dataset()
target_chunks = {"time": -1, "yt_ocean": 200, "xt_ocean": 400}  # ~45 chunks, ~230 MB each
rechunked = rechunk(
    ds_input,
    target_chunks=target_chunks,
    max_mem="25GB",  # memory per worker
    target_store=zarr_path,
    temp_store=f"{zarr_path}.tmp",  # temporary storage
)

rechunked.execute()
print(f"✅ Saved rechunked Zarr file to: {zarr_path}")

# Clean up temp store
import shutil
shutil.rmtree(f"{zarr_path}.tmp")

🗑️  Removing existing /scratch/m35/nm5072/TAS_thresholds/SH_MLD_ready.zarr
🗑️  Removing existing /scratch/m35/nm5072/TAS_thresholds/SH_MLD_ready.zarr.tmp
💾 Rechunking with rechunker...


/g/data/xp65/public/apps/med_conda/envs/analysis3-25.07/lib/python3.11/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 32.54 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [ ]:
ds

In [ ]:
ds = xr.open_zarr(zarr_path)

In [ ]:
# === Calculate climatology ===
Seas = MHW.smoothedClima_mhw(ds)
Seas.to_netcdf(output_dir / "om2_025_MLT_clim_SH.nc")

In [ ]:
# === Calculate threshold ===
Thresh = MHW.smoothedThresh_mhw(ds)
Thresh.to_netcdf(output_dir / "om2_025_MLT_thresh_SH.nc")

print("✅ Done.")

In [ ]:
clim = xr.open_dataset(output_dir / "om2_025_MLT_clim_SH.nc")
threshhold = xr.open_dataset(output_dir / "om2_025_MLT_thresh_SH.nc")

In [ ]:
threshhold

In [ ]:
clim.temp.isel(yt_ocean=150,xt_ocean=100).plot()
threshhold.temp.isel(yt_ocean=150,xt_ocean=100).plot()